# Gemini RAG 기본 구조 : Embedding / Ventor Search / RAG Engine

### 텍스트 임베딩( Vertex AI Text Embeddings)
- 1. 개요
비정형 텍스트를 의미 기반의 고차원 수치 벡터로 변환하는 기술입니다. 단순히 단어의 일치를 넘어 문맥적 유사성을 수학적으로 계산할 수 있게 합니다.
- 2. 핵심 모델 (2024~2025 기준)
text-embedding-004: 최신 영어 전용 모델. 가변 차원(Matryoshka) 지원으로 효율성 극대화.
text-multilingual-embedding-002: 한국어 포함 100개 이상 언어 지원. 다국어 RAG 구축 시 표준.
- 3. 주요 파라미터 및 지표
Dimension (차원): 벡터의 길이 (보정 가능). 보통 768 또는 1024 차원 사용.
Cosine Similarity: 두 벡터 간의 각도를 측정하여 유사도 판별 (1에 가까울수록 유사).
Task Type: 검색(RETRIEVAL_QUERY), 문서 색인(RETRIEVAL_DOCUMENT), 분류 등 목적 명시 가능.
- 4. 활용 워크플로우 (RAG 기준)
Chunking: 방대한 문서를 작은 단위로 분할.
Embedding: 분할된 텍스트를 Vertex AI 모델을 통해 벡터로 변환.
Vector DB 저장: 변환된 값을 전용 DB(Vertex AI Search 등)에 저장.
Retrieval: 사용자 질문을 임베딩하여 DB에서 가장 유사한 문서 추출.
- 5. 핵심 장점
Managed Service: 인프라 관리 없이 API 호출만으로 확장 가능한 성능 확보.
Google Search 급 품질: 구글 검색 엔진에 적용된 최신 임베딩 기술 활용 가능.

In [1]:
from google import genai
from google.genai.types import EmbedContentConfig

# 클라이언트 초기화
client = genai.Client(vertexai=True, project="byounghwa-go", location="global")

# 임베딩할 텍스트
texts_to_embed = [
    "운전면허증을 어떻게 받나요?",
    "운전면허증은 얼마나 오랫동안 유효한가요?",
    "운전면허 지식 테스트 학습 가이드"
]

# (Optional) 임베딩 설정 구성
embedding_config = EmbedContentConfig(
    task_type="RETRIEVAL_DOCUMENT",
    output_dimensionality=768, # 차원을 768로 지정하여 저장 공간 절약 가능
)

# 임베딩 생성 요청
response = client.models.embed_content(
    model="gemini-embedding-001",
    contents=texts_to_embed,
    # config=embedding_config, 설정하지 않을 시 Default 차원: 3072
)

# 결과 출력
print(f"생성된 임베딩 개수: {len(response.embeddings)}")

print(response.embeddings)

생성된 임베딩 개수: 3
[ContentEmbedding(
  statistics=ContentEmbeddingStatistics(
    token_count=13.0,
    truncated=False
  ),
  values=[
    0.010005897842347622,
    -0.006901226006448269,
    0.02454659901559353,
    -0.003643216099590063,
    -0.011352716945111752,
    <... 3067 more items ...>,
  ]
), ContentEmbedding(
  statistics=ContentEmbeddingStatistics(
    token_count=19.0,
    truncated=False
  ),
  values=[
    -0.010713272728025913,
    0.02316771261394024,
    0.0051171910017728806,
    -0.027457349002361298,
    -0.0030599648598581553,
    <... 3067 more items ...>,
  ]
), ContentEmbedding(
  statistics=ContentEmbeddingStatistics(
    token_count=12.0,
    truncated=False
  ),
  values=[
    0.006769051309674978,
    0.007541355676949024,
    0.031410906463861465,
    -0.01385351363569498,
    -0.016014080494642258,
    <... 3067 more items ...>,
  ]
)]


### 이미지 임베딩
텍스트 임베딩이 글의 '의미'를 숫자로 바꾼다면, Vertex AI 이미지 임베딩(Image Embeddings)은 이미지의 '시각적 특징'을 수치 벡터로 변환하는 기술입니다.
노트북 소스 문서에 추가하기 좋게 핵심만 정리해 드립니다.

- 1. 개요
이미지 내의 객체, 색상, 구도, 스타일 등 시각적 정보를 고차원 벡터로 변환합니다. 이를 통해 "이미지를 수치적으로 비교"할 수 있게 됩니다.
- 2. 핵심 모델: Multimodal Embedding
Vertex AI는 주로 Multimodal Embedding 모델을 사용합니다.
특징: 이미지와 텍스트를 동일한 벡터 공간 안에 위치시킵니다.
장점: "파란색 운동화"라는 텍스트 벡터와 실제 파란 운동화 이미지 벡터가 가깝게 위치하게 되어, 텍스트로 이미지를 찾는 시각적 검색(Visual Search)이 가능해집니다.
- 3. 주요 활용 사례
시각적 유사도 검색: 특정 제품 사진을 올리면 디자인이 유사한 다른 상품을 추천.
이미지 분류 및 태깅: 별도의 레이블링 없이도 이미지를 자동으로 카테고리화.
텍스트-이미지 검색 (Cross-modal): 키워드 검색을 통해 라이브러리 내 적절한 이미지 추출.
이상 탐지: 정상 이미지의 벡터 범위를 벗어난 불량품이나 이상 데이터 감지.
- 4. 기술적 특징
입력: 이미지 파일 (PNG, JPEG 등) 또는 텍스트 쿼리.
출력: 보통 128, 256, 512, 1408 차원의 벡터 (설정에 따라 가변적).
성능: 수억 장의 이미지도 벡터 DB(Vertex AI Vector Search)와 결합하면 밀리초(ms) 단위로 검색 가능.
- 5. 핵심 가치
"백문이 불여일견"을 데이터화하는 기술입니다. 텍스트로 설명하기 힘든 디자인, 패턴, 분위기 등의 추상적 시각 정보를 정교한 숫자로 다룰 수 있게 해줍니다.

In [2]:
import os
from google.cloud import storage
import vertexai
from vertexai.vision_models import Image, MultiModalEmbeddingModel

# 1. 환경 설정
PROJECT_ID = "byounghwa-go"
LOCATION = "asia-northeast3"
BUCKET_NAME = f"{PROJECT_ID}-rag-bucket" # 중복 방지를 위한 버킷 이름
FILE_NAME = "image.jpg"

# 2. Cloud Storage 작업 (버킷 생성 및 업로드)
storage_client = storage.Client(project=PROJECT_ID)

# 버킷이 없으면 생성
bucket = storage_client.lookup_bucket(BUCKET_NAME)
if not bucket:
    bucket = storage_client.create_bucket(BUCKET_NAME, location=LOCATION)
    print(f"버킷 생성됨: {BUCKET_NAME}")

# 파일 업로드 (image.jpg -> 버킷 내 image.jpg)
blob = bucket.blob(FILE_NAME)
blob.upload_from_filename(FILE_NAME)
gcs_uri = f"gs://{BUCKET_NAME}/{FILE_NAME}"
print(f"업로드 완료: {gcs_uri}")

# 3. Vertex AI 임베딩 작업
vertexai.init(project=PROJECT_ID, location=LOCATION)

# 모델 로드 및 임베딩 추출
model = MultiModalEmbeddingModel.from_pretrained("multimodalembedding@001")
image_input = Image.load_from_file(gcs_uri)

embeddings = model.get_embeddings(
    image=image_input,
    contextual_text="Female astronaut photo", # 이미지와 관련된 설명
    dimension=1408
)

# 4. 결과 출력
print("-" * 50)
print("성공적으로 임베딩을 추출했습니다!")
print(f"이미지 벡터(앞 5개): {embeddings.image_embedding[:5]}")
print(f"텍스트 벡터(앞 5개): {embeddings.text_embedding[:5]}")
print("-" * 50)

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.resourcemanager_v3 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.resourcemanager_v3 past that date.
  warnings.warn(message, FutureWarning)


버킷 생성됨: byounghwa-go-rag-bucket
업로드 완료: gs://byounghwa-go-rag-bucket/image.jpg


/opt/conda/lib/python3.10/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
/opt/conda/lib/python3.10/site-packages/vertexai/vision_models/_vision_models.py:154: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


--------------------------------------------------
성공적으로 임베딩을 추출했습니다!
이미지 벡터(앞 5개): [0.0222626552, 0.0640854612, 0.0055154264, 0.0341656655, 0.0206377879]
텍스트 벡터(앞 5개): [0.0285653342, -0.0075667277, 0.0114342654, 0.00348207285, 0.0486590303]
--------------------------------------------------


### Vertex AI Vector Search
임베딩이 데이터를 '숫자 좌표'로 만드는 과정이라면, Vertex AI Vector Search(구 Matching Engine)는 그 좌표들 사이에서 가장 가까운 이웃을 빛의 속도로 찾아내는 검색 엔진입니다.
- 1. 개요
수십억 개의 고차원 벡터 데이터셋에서 유사한 항목을 밀리초(ms) 단위로 찾아내는 고성능 벡터 데이터베이스 서비스입니다. Google 검색과 YouTube 추천에 사용되는 것과 동일한 알고리즘(ScaNN)을 기반으로 합니다.
- 2. 핵심 메커니즘: ANN (Approximate Nearest Neighbor)
모든 데이터와 일일이 대조하는 대신, 수학적 기법을 통해 유사할 가능성이 높은 후보군을 빠르게 좁혀 검색합니다.
고가용성: 대규모 트래픽에서도 낮은 지연 시간(Low Latency) 유지.
확장성: 수억~수십억 개의 벡터를 안정적으로 관리.
- 3. 주요 기능 및 용어
Index: 벡터 데이터가 저장되고 검색 가능하도록 구조화된 단위.
Index Endpoint: 인덱스를 배포하여 실제 API 호출을 받는 서버 지점.
Filtering: 벡터 유사도뿐만 아니라 '가격', '카테고리' 등 일반 속성(Metadata)을 결합해 검색 결과 제한.
Crowding: 검색 결과에 동일한 작성자나 유사한 카테고리만 도배되지 않도록 다양성 조절.
- 4. 전체 워크플로우
Embed: 텍스트/이미지를 Vertex AI Embedding 모델로 벡터화.
Ingest: 생성된 벡터를 Google Cloud Storage에 업로드.
Create Index: 업로드된 데이터를 바탕으로 검색 인덱스 생성.
Deploy: 인덱스를 엔드포인트에 배포.
Query: 사용자 질문을 벡터로 변환 후 Vector Search에 던져 유사 결과 획득.
- 5. 한 줄 정의
"임베딩이 데이터의 '지도'를 그린다면, Vector Search는 그 지도에서 가장 가까운 목적지를 찾아주는 '내비게이션'입니다."
-   Tip: 단순히 유사도만 보는 것이 아니라, 내부 메타데이터 필터링을 함께 사용해야 실제 서비스(예: "서울 지역"에 있는 "운동화" 중 유사한 것 찾기)에서 강력한 힘을 발휘합니다.

In [3]:
# Vertex AI Embeddings를 활용한 임베딩 생성 
import json
import os
from typing import List, Dict, Optional
from faq import faq_data_raw

import vertexai
from google import genai
from google.genai.types import EmbedContentConfig

PROJECT_ID = "byounghwa-go"
LOCATION = "asia-northeast3"

# 클라이언트 초기화
vertexai.init(project=PROJECT_ID, location=LOCATION)
genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

# Vertex AI Embeddings 모델을 활용한 임베딩 생성
def get_embedding_vector(texts: List[str], task_type: str) -> List[List[float]]:
   
    # 1. 임베딩 설정 구성
    embedding_config = EmbedContentConfig(
        task_type=task_type,
        output_dimensionality=768
    )
   
    # 2. 임베딩 요청
    response = genai_client.models.embed_content(
        model="gemini-embedding-001",
        contents=texts,
        config=embedding_config,
    )
    return [emb.values for emb in response.embeddings]

In [4]:
# 데이터 임베딩 및 JSONL 파일 생성
texts_to_embed = [item["text"] for item in faq_data_raw]
document_embeddings = get_embedding_vector(texts_to_embed, "RETRIEVAL_DOCUMENT")

file_path = "faq_embeddings.json"
vector_datapoints = []
for item, emb in zip(faq_data_raw, document_embeddings):
    datapoint = {
        "id": item["id"],
        "embedding": emb,
    }
    vector_datapoints.append(datapoint)
   
with open(file_path, "w") as f:
    for dp in vector_datapoints:
        f.write(json.dumps(dp) + "\n")

In [5]:
# GCS 업로드
storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)

blob_path = "Vector_Search/FAQ/faq_embeddings.json"
blob = bucket.blob(blob_path)
blob.upload_from_filename(file_path)

print(f"GCS 업로드 완료: gs://{BUCKET_NAME}/{blob_path}")

GCS 업로드 완료: gs://byounghwa-go-rag-bucket/Vector_Search/FAQ/faq_embeddings.json


In [6]:
# 색인 구축 및 엔드포인트 배포 
from google.cloud import aiplatform

# AI Platform(Vertex AI의 이전 이름) 클라이언트 초기화
aiplatform.init(project=PROJECT_ID, location=LOCATION)

INDEX_DISPLAY_NAME = "faq_index"
GCS_INPUT_URI = f"gs://{BUCKET_NAME}/Vector_Search/FAQ"
DEPLOYED_INDEX_ID = "faq_index_deployed"

faq_index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
    display_name=INDEX_DISPLAY_NAME,
    contents_delta_uri=GCS_INPUT_URI,
    description="FAQ 데이터에 대한 벡터DB 입니다.",
    dimensions=768,
    approximate_neighbors_count=50,
    leaf_node_embedding_count=10,
    leaf_nodes_to_search_percent=20,
    distance_measure_type=aiplatform.matching_engine.matching_engine_index_config.DistanceMeasureType.DOT_PRODUCT_DISTANCE,
    index_update_method="BATCH_UPDATE", # 일괄 업데이트 방식 사용
)

# 색인 엔드포인트 생성 
faq_index_endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
    display_name=f"faq_index_endpoint",
    public_endpoint_enabled=True,
)

# 엔드포인트에 색인 배포 
faq_index_endpoint.deploy_index(
    index=faq_index,
    deployed_index_id=DEPLOYED_INDEX_ID,
    min_replica_count=1,
)
# 장시간 소요 : 약 27분 소요됨

Creating MatchingEngineIndex
Create MatchingEngineIndex backing LRO: projects/810458263508/locations/asia-northeast3/indexes/2968650608669622272/operations/1074442212906893312
MatchingEngineIndex created. Resource name: projects/810458263508/locations/asia-northeast3/indexes/2968650608669622272
To use this MatchingEngineIndex in another session:
index = aiplatform.MatchingEngineIndex('projects/810458263508/locations/asia-northeast3/indexes/2968650608669622272')
Creating MatchingEngineIndexEndpoint
Create MatchingEngineIndexEndpoint backing LRO: projects/810458263508/locations/asia-northeast3/indexEndpoints/8168060776515371008/operations/9035117504237666304
MatchingEngineIndexEndpoint created. Resource name: projects/810458263508/locations/asia-northeast3/indexEndpoints/8168060776515371008
To use this MatchingEngineIndexEndpoint in another session:
index_endpoint = aiplatform.MatchingEngineIndexEndpoint('projects/810458263508/locations/asia-northeast3/indexEndpoints/8168060776515371008'

resource name: projects/810458263508/locations/asia-northeast3/indexEndpoints/8168060776515371008

In [48]:
#  시맨틱 검색 실행 
PROJECT_ID = "byounghwa-go"
LOCATION = "asia-northeast3"

# 색인 엔드포인트 ID (Vertex AI 콘솔의 벡터 검색 --> '색인 엔드포인트'에서 확인할 수 있다. asia-northeast3 리전으로 변경해서 확인)
INDEX_ENDPOINT_ID = "8168060776515371008"  # 자신의 값으로 맞게 수정한다

# 배포된 색인 ID (Endpoint에 배포할 때 지정한 고유 ID)
DEPLOYED_INDEX_ID = "faq_index_deployed"

# 유사도 검색에 사용할 사용자 질문
QUERY_TEXT = "고객 센터는 몇 시까지 운영하나요?"

# QUERY_TEXT = "계정은 어떻게 만드나요?"

# Google GenAI 클라이언트 초기화
genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

# 기존 MatchingEngineIndexEndpoint 객체 로드
faq_index_endpoint = aiplatform.MatchingEngineIndexEndpoint(
    index_endpoint_name=INDEX_ENDPOINT_ID 
)

# 텍스트 임베딩 생성
embedding_config = EmbedContentConfig(
    task_type="RETRIEVAL_QUERY",
    output_dimensionality=768,    # 색인 생성 시 사용된 차원과 반드시 일치
)

response_emb = genai_client.models.embed_content(
    model="gemini-embedding-001",
    contents=[QUERY_TEXT],
    config=embedding_config,
)

# 임베딩 벡터 추출
query_emb: List[float] = response_emb.embeddings[0].values

# 유사도 검색 수행
response = faq_index_endpoint.find_neighbors(
    deployed_index_id=DEPLOYED_INDEX_ID,
    queries=[query_emb],
    num_neighbors=3,
)

# 결과
for neighbor in response[0]:
        print(f"ID: {neighbor.id}, 거리(Distance): {neighbor.distance:.4f}")

ID: 31, 거리(Distance): 0.2806
ID: 32, 거리(Distance): 0.2542
ID: 33, 거리(Distance): 0.2447


## RAG Engine
Vertex AI RAG Engine은 앞서 정리한 임베딩과 벡터 검색, 그리고 LLM을 하나로 합쳐 **'우리 회사 맞춤형 AI'**를 가장 빠르게 만들 수 있게 해주는 완전 관리형 프레임워크입니다.
- 1. 개요
RAG(Retrieval-Augmented Generation) 구현에 필요한 모든 복잡한 과정을 자동화한 서비스입니다. 데이터 소스 연결부터 검색, 답변 생성까지의 파이프라인을 코드 몇 줄로 구축할 수 있게 해줍니다.
- 2. 핵심 구성 요소 (The 3 Pillars)
Data Source: Google Drive, Cloud Storage, BigQuery 등 기업 데이터를 직접 연결.
RAG Corpus: 업로드된 문서를 자동으로 분할(Chunking)하고 임베딩하여 저장하는 관리형 저장소.
Retrieval & Generation: 질문에 맞는 최적의 문맥을 찾아 LLM(Gemini)에게 전달하고 최종 답변 생성.
- 3. RAG Engine의 차별점
No-Infrastructure: 별도의 벡터 DB 서버를 관리하거나 인덱싱 로직을 짤 필요가 없습니다.
자동화된 전처리: PDF, HTML, TXT 등 다양한 문서 형식을 알아서 읽고 쪼개줍니다.
Grounding (근거 제시): LLM이 답변할 때 어떤 문서의 몇 페이지를 참고했는지 출처를 명확히 제시하여 환각(Hallucination) 현상을 방지합니다.
간편한 API: 복잡한 LangChain 코드 없이 RagCorpus, RagFile 객체 모델을 통해 직관적으로 조작합니다.
- 4. 구축 프로세스 (Quick Workflow)
Corpus 생성: 데이터를 담을 가상 저장소 생성.
파일 업로드: 문서 데이터를 Corpus에 삽입 (자동 임베딩 진행).
질의(Query): retrieval_query API를 호출하여 답변 획득.
- 5. 한 줄 정의
"임베딩, 벡터 검색, LLM을 개별적으로 조립할 필요 없이, 데이터만 넣으면 바로 작동하는 'RAG 전용 조립 키트'입니다."

- Tip: 직접 커스텀한 정교한 로직이 필요하다면 Vector Search를 개별적으로 쓰고, 빠른 프로토타이핑과 안정적인 관리가 우선이라면 RAG Engine을 선택하는 것이 전략적입니다.

In [49]:
# 코퍼스 생성 
from vertexai import rag
import vertexai

# 클라이언트 초기화
vertexai.init(project="byounghwa-go", location="asia-northeast3")

# 벡엔드 임베딩 모델 설정
backend_config = rag.RagVectorDbConfig(
    rag_embedding_model_config=rag.RagEmbeddingModelConfig(
        vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
            publisher_model="publishers/google/models/text-multilingual-embedding-002"
        )
    )
)

corpus = rag.create_corpus(
    display_name="Sample_Corpus",
    description="산업 IoT용 엣지컴퓨팅 기술 동향 데이터입니다.",
    backend_config=backend_config,
)
print(corpus)

RagCorpus(name='projects/810458263508/locations/asia-northeast3/ragCorpora/2305843009213693952', display_name='Sample_Corpus', description='산업 IoT용 엣지컴퓨팅 기술 동향 데이터입니다.', vertex_ai_search_config=None, backend_config=RagVectorDbConfig(vector_db=RagManagedDb(), rag_embedding_model_config=RagEmbeddingModelConfig(vertex_prediction_endpoint=VertexPredictionEndpoint(endpoint=None, publisher_model='projects/byounghwa-go/locations/asia-northeast3/publishers/google/models/text-multilingual-embedding-002', model=None, model_version_id=None))), encryption_spec=)


In [50]:
# 로컬 pdf 파일 업로드 : PDF파일 사이즈 최대 50MB로 제한됨

# 앞에서 출력된 name 값으로 수정한다
corpus_name='projects/810458263508/locations/asia-northeast3/ragCorpora/2305843009213693952'

path='./산업_IoT용_엣지컴퓨팅_기술동향.pdf'

rag_file = rag.upload_file(
    corpus_name=corpus_name,
    path=path,
   
    # 아래 설정은 옵션입니다.
    transformation_config=rag.TransformationConfig(
        chunking_config=rag.ChunkingConfig(
            chunk_size=512,
            chunk_overlap=100,
        ),
    ),
)
print(rag_file)

# Vertex AI --> Vertex AI RAG Engine 에 Sample_Corpus 생성 확인

RagFile(name='projects/810458263508/locations/asia-northeast3/ragCorpora/2305843009213693952/ragFiles/5675355169461813336', display_name='vertex-2026-04-13-07-51-02-e740e', description=None)


In [ ]:
# # Google Cloud Storage, Google Drive 파일 업로드 : 실습 생략
# corpus_name='projects/<프로젝트 넘버>/locations/asia-northeast3/ragCorpora/<코퍼스ID>'
# paths=['https://drive.google.com/file/d/<파일ID>', 'gs://my-data-bucket/VertexAI_RAG_Engine/산업_IoT용_엣지컴퓨팅_기술동향.pdf']

# # 파일 import
# rag_files = rag.import_files(
#     corpus_name=corpus_name,
#     paths=paths,


#     transformation_config=rag.TransformationConfig(
#         chunking_config=rag.ChunkingConfig(
#             chunk_size=512,
#             chunk_overlap=100,
#         ),
#     ),
#     max_embedding_requests_per_min=1000, # import_files에서만 지정 가능합니다.
# )

# print(rag_files)

In [54]:
# 검색 
response = rag.retrieval_query(
    rag_resources=[
        rag.RagResource(
            rag_corpus=corpus_name,
        )
    ],
    text="엣지 컴퓨팅 기술 동향 알려줘",
    rag_retrieval_config=rag.RagRetrievalConfig(
        top_k=3,
        filter=rag.utils.resources.Filter(vector_distance_threshold=0.5),
    ),
)
print(response)

contexts {
  contexts {
    source_uri: "vertex-2026-04-13-07-51-02-e740e"
    text: "가트너의 ‘Top 10 Strategic Technology Trend 2019’ 보고서\r\n에 따르면 지능정보사회의 혁신기술 동향으로 10대 전략 기\r\n술을 발표하였는데, 이 중에는 Autonomous Things(자율 사\r\n물), AI-Driven Development(인공지능 주도 개발), Digital \r\nTwin(디지털 트윈), Empowered Edge(자율성을 가진 엣지) \r\n등이 포함된다. 이 중에서 엣지 컴퓨팅 기술은 정보 처리 및\r\n콘텐츠 수집과 전달이 센서/디바이스 및 사용자와 인접한 곳\r\n에서 처리되는 컴퓨팅 기술을 말하며, 이는 데이터 트래픽\r\n과 실시간 처리를 목적으로 로컬에서 데이터를 처리한다[3]. 이러한 엣지 컴퓨팅 기술은 스마트 팩토리, 스마트 공장\r\n을 포함한 산업 IoT분야에서 폭발적으로 증가할 것으로 예\r\n상되는 데이터의 처리 시간 단축, 데이터 보안 강화, 데이터\r\n전송 비용 감소를 위하여 많은 글로벌 기업에서 관련 연구\r\n개발이 활발히 진행되고 있다. 본 논문에서는 산업 IoT 분야\r\n에서 데이터의 효율적인 프로세싱 및 지능화 제공을 목표로\r\n개발되고 있는 엣지 컴퓨팅 기술에 대하여 소개한다. Ⅱ."
    source_display_name: "vertex-2026-04-13-07-51-02-e740e"
    score: 0.14844127052207423
    chunk {
      text: "가트너의 ‘Top 10 Strategic Technology Trend 2019’ 보고서\r\n에 따르면 지능정보사회의 혁신기술 동향으로 10대 전략 기\r\n술을 발표하였는데, 이 중에는 Autonomous Things(자율 사\r\n물), AI-Driven Development(인공지능 주도 개발), Digital \

In [56]:
#  답변 생성 
from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project='byounghwa-go', location='global')

rag_retrieval_tool = [
    types.Tool(
      retrieval=types.Retrieval(
        vertex_rag_store=types.VertexRagStore(
            rag_resources=[
                types.VertexRagStoreRagResource(
                rag_corpus=corpus_name
                )
            ],
            rag_retrieval_config=rag.RagRetrievalConfig(
                top_k=3,
                filter=rag.utils.resources.Filter(vector_distance_threshold=0.5)
            ),          
        )
      )
    )
]

response = client.models.generate_content(
    model = "gemini-3-flash-preview",
    contents = "엣지 컴퓨팅 기술 동향 알려줘",
    config = types.GenerateContentConfig(tools=rag_retrieval_tool)
)

print(response.text)

엣지 컴퓨팅은 데이터가 발생하는 센서나 디바이스 인접 곳에서 정보를 처리하는 기술로, 최근 동향은 다음과 같습니다.

*   **지능형 엣지 컴퓨팅으로의 진화:** 단순 데이터 처리를 넘어 클라우드 및 인공지능(AI) 기술과 통합되고 있으며, 실시간 추론, 예측, 자율 기능을 수행하는 지능형 서비스로 발전하고 있습니다.
*   **기술 분류 및 주도 기업:** 아마존, 마이크로소프트, 구글 등 클라우드 기업 중심의 플랫폼 기술과 삼성전자, DELL, Cisco 등 장비 제조업체 중심의 산업용 IoT 플랫폼 기술, 그리고 오픈소스 및 하드웨어 플랫폼 등으로 구분되어 개발되고 있습니다.
*   **산업 IoT 분야 확산:** 스마트 팩토리 등 데이터 처리가 급증하는 산업 현장에서 데이터 전송 비용 절감, 보안 강화, 실시간 처리를 위해 연구개발 및 활용이 활발히 진행 중입니다.
